In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -U \
  torch torchvision torchaudio \
  --index-url https://download.pytorch.org/whl/cu121


In [ ]:
!pip install -U \
  transformers \
  datasets \
  accelerate \
  peft \
  bitsandbytes \
  evaluate \
  rouge-score \
  bert-score \
  sentencepiece \
  openpyxl


In [ ]:
import torch
import torchvision
import transformers

from transformers import Seq2SeqTrainer, PreTrainedModel




In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Transformers and datasets
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    TrainerCallback,
    GenerationConfig
)
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

# Evaluation metrics
import evaluate
from rouge_score import rouge_scorer
from bert_score import score as bert_score

# Set seeds for reproducibility
import random
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)



### 2.2 Load Dataset

In [ ]:
# Load training data
try:
    train_df = pd.read_csv('medical_dialogue_train.csv')
    print(f"✓ Training data loaded: {train_df.shape}")
except FileNotFoundError:
    print("❌ Training file not found. Please upload the dataset.")
    raise

# Load validation data
try:
    val_df = pd.read_excel('medical_dialogue_validation.xlsx')
    print(f"✓ Validation data loaded: {val_df.shape}")
except FileNotFoundError:
    print("❌ Validation file not found.")
    raise

# Load test data
try:
    test_df = pd.read_excel('medical_dialogue_test.xlsx')
    print(f"✓ Test data loaded: {test_df.shape}")
except FileNotFoundError:
    print("❌ Test file not found.")
    raise

print("\n" + "="*80)
print("Dataset Summary:")
print("="*80)
print(f"Training samples: {len(train_df):,}")
print(f"Validation samples: {len(val_df):,}")
print(f"Test samples: {len(test_df):,}")
print(f"Total samples: {len(train_df) + len(val_df) + len(test_df):,}")
print("="*80)

### 2.3 Data Exploration

### 2.4 Text Length Analysis

In [ ]:
# Calculate text lengths
train_df['dialogue_length'] = train_df['dialogue'].str.len()
train_df['soap_length'] = train_df['soap'].str.len()

# Statistics
print("Dialogue length statistics:")
print(train_df['dialogue_length'].describe())
print("\nSOAP summary length statistics:")
print(train_df['soap_length'].describe())

# Visualize distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(train_df['dialogue_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Dialogue Length (characters)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Dialogue Lengths')
axes[0].axvline(train_df['dialogue_length'].mean(), color='red', linestyle='--', label='Mean')
axes[0].legend()

axes[1].hist(train_df['soap_length'], bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1].set_xlabel('SOAP Summary Length (characters)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of SOAP Summary Lengths')
axes[1].axvline(train_df['soap_length'].mean(), color='red', linestyle='--', label='Mean')
axes[1].legend()

plt.tight_layout()
plt.show()

# Compression ratio
compression_ratio = train_df['dialogue_length'].mean() / train_df['soap_length'].mean()
print(f"\nAverage compression ratio: {compression_ratio:.2f}x")

## 3. Data Preprocessing

### 3.1 Clean and Prepare Data

In [ ]:
def clean_text(text):
    """Clean text by removing extra whitespace and newlines"""
    if pd.isna(text):
        return ""
    # Remove excessive whitespace
    text = ' '.join(text.split())
    return text.strip()

# Clean all datasets
print("Cleaning datasets...")
for df in [train_df, val_df, test_df]:
    df['dialogue'] = df['dialogue'].apply(clean_text)
    df['soap'] = df['soap'].apply(clean_text)

# Remove empty entries
train_df = train_df[(train_df['dialogue'].str.len() > 0) & (train_df['soap'].str.len() > 0)]
val_df = val_df[(val_df['dialogue'].str.len() > 0) & (val_df['soap'].str.len() > 0)]
test_df = test_df[(test_df['dialogue'].str.len() > 0) & (test_df['soap'].str.len() > 0)]

print(f"✓ Cleaned training samples: {len(train_df):,}")
print(f"✓ Cleaned validation samples: {len(val_df):,}")
print(f"✓ Cleaned test samples: {len(test_df):,}")

### 3.2 Convert to HuggingFace Dataset Format

In [ ]:
# Create dataset dictionary
dataset_dict = DatasetDict({
    'train': Dataset.from_pandas(train_df[['dialogue', 'soap']].reset_index(drop=True)),
    'validation': Dataset.from_pandas(val_df[['dialogue', 'soap']].reset_index(drop=True)),
    'test': Dataset.from_pandas(test_df[['dialogue', 'soap']].reset_index(drop=True))
})

print("✓ Dataset converted to HuggingFace format:")
print(dataset_dict)

## 4. Model Selection and Configuration

### 4.1 Select Pre-trained Model



In [ ]:
# Model configuration
MODEL_NAME = "google/flan-t5-base"  # ~250M parameters
MAX_INPUT_LENGTH = 512  # Maximum length for input dialogue
MAX_TARGET_LENGTH = 256  # Maximum length for SOAP summary

print(f"Selected model: {MODEL_NAME}")
print(f"Max input length: {MAX_INPUT_LENGTH}")
print(f"Max target length: {MAX_TARGET_LENGTH}")

### 4.2 Load Tokenizer and Model

In [ ]:
# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("✓ Tokenizer loaded")

# Load base model

base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print(f"✓ Model loaded successfully!")
print(f"Model parameters: {base_model.num_parameters():,}")

### 4.3 Test Baseline Model (Before Fine-tuning)



In [ ]:
def generate_summary(model, tokenizer, dialogue, max_length=256):
    """
    Generate SOAP summary from dialogue using the model
    """
    # Create instruction-based prompt
    prompt = f"""Summarize the following medical dialogue in SOAP (Subjective, Objective, Assessment, Plan) format:

Dialogue:
{dialogue}

SOAP Summary:"""

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=4,
            early_stopping=True,
            temperature=0.7,
            do_sample=False
        )

    # Decode
    summary = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return summary

# Test baseline on 2 examples
print("\n" + "="*100)
print("BASELINE MODEL PERFORMANCE (Before Fine-tuning)")
print("="*100)

for i in range(min(2, len(test_df))):
    test_dialogue = test_df.iloc[i]['dialogue']
    reference_soap = test_df.iloc[i]['soap']

    print(f"\n\n{'='*100}")
    print(f"EXAMPLE {i+1}")
    print(f"{'='*100}")
    print(f"\nDIALOGUE (truncated):")
    print(test_dialogue[:300] + "..." if len(test_dialogue) > 300 else test_dialogue)

    print(f"\n{'-'*100}")
    print(f"REFERENCE SOAP:")
    print(reference_soap)

    print(f"\n{'-'*100}")
    print(f"BASELINE GENERATED SOAP:")
    baseline_summary = generate_summary(base_model, tokenizer, test_dialogue)
    print(baseline_summary)

print("\n" + "="*100)

## 5. Data Tokenization and Preparation

### 5.1 Create Preprocessing Function

In [ ]:
def preprocess_function(examples):
    """
    Preprocess the data for fine-tuning:
    - Add instruction prefix to dialogues
    - Tokenize inputs and targets
    """
    # Create instruction-based prompts
    inputs = [
        f"Summarize the following medical dialogue in SOAP format: {dialogue}"
        for dialogue in examples['dialogue']
    ]

    # Tokenize inputs
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding="max_length"
    )

    # Tokenize targets (SOAP summaries)
    labels = tokenizer(
        examples['soap'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

### 5.2 Apply Tokenization

In [ ]:
# Tokenize datasets

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset_dict["train"].column_names,
    desc="Tokenizing"
)

print("\n✓ Tokenization complete!")
print("\nTokenized dataset structure:")
print(tokenized_datasets)

## 6. Fine-tuning with LoRA (Parameter-Efficient Fine-Tuning)

### 6.1 Configure LoRA

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=16,  # Rank of the update matrices
    lora_alpha=32,  # Scaling factor
    target_modules=["q", "v"],  # Which layers to apply LoRA to
    lora_dropout=0.05,  # Dropout probability
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

# Apply LoRA to the model
print("Applying LoRA to model...")
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
print("\n✓ LoRA applied successfully!")
model.print_trainable_parameters()

### 6.2 Create Custom Loss Logging Callback



In [ ]:
class DetailedLossLoggingCallback(TrainerCallback):
    """Custom callback to properly log training loss at every step"""

    def on_log(self, args, state, control, logs=None, **kwargs):
        """Called when logging occurs"""
        if logs:
            if 'loss' in logs:
                print(f"  Step {state.global_step}: Training Loss = {logs['loss']:.4f}")
            if 'eval_loss' in logs:
                print(f"  Step {state.global_step}: Validation Loss = {logs['eval_loss']:.4f}")

    def on_step_end(self, args, state, control, **kwargs):
        """Print progress every 50 steps"""
        if state.global_step % 50 == 0:
            if hasattr(state, 'log_history') and state.log_history:
                last_log = state.log_history[-1]
                if 'loss' in last_log:
                    print(f"  → Step {state.global_step}: Recent loss = {last_log['loss']:.4f}")

print("✓ Custom callback created")

### 6.3 Set Training Arguments



In [ ]:
#  Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/drive/MyDrive/model/medical_soap_finetuned",

    # ========== Core Training Hyperparameters ==========
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,  # Effective batch size = 16

    # ========== Optimization (FIXED) ==========
    learning_rate=1e-4,  # REDUCED from 3e-4 for stability
    weight_decay=0.01,
    warmup_ratio=0.1,    # ADDED: 10% warmup for stability
    max_grad_norm=1.0,   # ADDED: Gradient clipping

    # ========== Mixed Precision (CHANGED FOR TESTING) ==========
    fp16=False,  # CHANGED: Test with FP32 first to rule out precision issues
    # fp16=True,  # Re-enable AFTER confirming FP32 works

    # ========== Evaluation and Logging (IMPROVED) ==========
    eval_strategy="steps",
    eval_steps=250,              # MORE FREQUENT (was 500)
    logging_steps=25,            # MORE FREQUENT (was 100)
    logging_first_step=True,     # ADDED: Log initial loss
    logging_strategy="steps",    # ADDED: Explicit strategy
    logging_nan_inf_filter=False,  # ADDED: See actual values
    save_steps=250,              # MORE FREQUENT (was 500)
    save_total_limit=3,

    # ========== Generation Settings ==========
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LENGTH,

    # ========== Other Settings ==========
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
    seed=42,

    # ========== Additional Stability Settings ==========
    dataloader_pin_memory=True,
    dataloader_num_workers=0,  # Avoid multiprocessing issues
)



### 6.4 Data Collator

In [ ]:
# Data collator for dynamic padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

print("✓ Data collator created")

### 6.5 Initialize Trainer with Callback

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
)


### 6.6 Start Fine-tuning



In [ ]:
train_result = trainer.train()

### 6.7 Save Fine-tuned Model

In [ ]:
# Save the fine-tuned model
output_dir = "/content/drive/MyDrive/model/finetuned_medical_soap_model"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"✓ Model saved to {output_dir}")

# Also save training arguments for reproducibility
import json
training_args_dict = training_args.to_dict()
with open(f"{output_dir}/training_args.json", 'w') as f:
    json.dump(training_args_dict, f, indent=2)

# Save training results
with open(f"{output_dir}/training_results.json", 'w') as f:
    json.dump(train_result.metrics, f, indent=2)

print("✓ Training arguments and results saved!")

## 7. Model Evaluation

### 7.1 Load Evaluation Metrics

In [ ]:
# Load ROUGE metric
rouge = evaluate.load('rouge')

print("✓ Evaluation metrics loaded successfully!")

### 7.2 Generate Predictions on Test Set

In [ ]:
# Generate predictions for test set
def generate_predictions(model, tokenizer, test_dataset, num_samples=None):
    """
    Generate predictions for the test dataset
    """
    if num_samples:
        indices = list(range(min(num_samples, len(test_dataset))))
        test_data = test_dataset.select(indices)
    else:
        test_data = test_dataset

    predictions = []
    references = []

    print(f"Generating predictions for {len(test_data)} samples...")

    for i, example in enumerate(test_data):
        dialogue = example['dialogue']
        reference = example['soap']

        # Generate summary
        prediction = generate_summary(model, tokenizer, dialogue, MAX_TARGET_LENGTH)

        predictions.append(prediction)
        references.append(reference)

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(test_data)} samples")

    return predictions, references


print("\nGenerating predictions on test set...")
predictions, references = generate_predictions(
    model,
    tokenizer,
    dataset_dict['test'],
    num_samples=100  # Evaluate on 100 samples

print(f"\n✓ Generated {len(predictions)} predictions")

### 7.3 Calculate ROUGE Scores



In [ ]:
# Calculate ROUGE scores
rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True
)

print("\n" + "="*80)
print("ROUGE SCORES (Fine-tuned Model)")
print("="*80)
print(f"ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"ROUGE-Lsum: {rouge_results['rougeLsum']:.4f}")
print("="*80)



### 7.4 Calculate BERTScore

In [ ]:
# Calculate BERTScore (using a subset for speed)

print("\n" + "="*80)
print("BERTScore (Semantic Similarity)")
print("="*80)
print(f"Precision: {P.mean():.4f}")
print(f"Recall: {R.mean():.4f}")
print(f"F1: {F1.mean():.4f}")
print("="*80)

### 7.5 SOAP Component Analysis

In [ ]:
def analyze_soap_components(text):
    """Check if text contains SOAP components"""
    text_lower = text.lower()

    components = {
        'subjective': any(kw in text_lower for kw in ['subjective', 'patient reports', 'complains of', 'symptoms']),
        'objective': any(kw in text_lower for kw in ['objective', 'physical exam', 'vital signs', 'examination']),
        'assessment': any(kw in text_lower for kw in ['assessment', 'diagnosis', 'impression', 'findings']),
        'plan': any(kw in text_lower for kw in ['plan', 'treatment', 'recommend', 'follow-up', 'prescription'])
    }

    return components

# Analyze SOAP components
component_presence = {'subjective': 0, 'objective': 0, 'assessment': 0, 'plan': 0}

for pred in predictions:
    components = analyze_soap_components(pred)
    for key, value in components.items():
        if value:
            component_presence[key] += 1

total_predictions = len(predictions)

print("\n" + "="*80)
print("SOAP Component Presence Analysis")
print("="*80)
for component, count in component_presence.items():
    percentage = (count / total_predictions) * 100
    print(f"{component.capitalize()}: {count}/{total_predictions} ({percentage:.1f}%)")
print("="*80)

### 7.6 Qualitative Analysis - Good Examples

In [ ]:
# Find examples with high ROUGE scores
individual_rouge_scores = []

for pred, ref in zip(predictions, references):
    score = rouge.compute(predictions=[pred], references=[ref])['rougeL']
    individual_rouge_scores.append(score)

# Get indices of top 3 examples
top_indices = np.argsort(individual_rouge_scores)[-3:][::-1]

print("\n" + "="*100)
print("GOOD EXAMPLES (High ROUGE-L Scores)")
print("="*100)

for rank, idx in enumerate(top_indices, 1):
    test_example = dataset_dict['test'][idx]

    print(f"\n\n{'='*100}")
    print(f"EXAMPLE {rank} - ROUGE-L: {individual_rouge_scores[idx]:.4f}")
    print(f"{'='*100}")

    print(f"\nDIALOGUE (truncated):")
    dialogue = test_example['dialogue']
    print(dialogue[:400] + "..." if len(dialogue) > 400 else dialogue)

    print(f"\n{'-'*100}")
    print(f"REFERENCE SOAP:")
    print(references[idx])

    print(f"\n{'-'*100}")
    print(f"GENERATED SOAP:")
    print(predictions[idx])

### 7.7 Qualitative Analysis - Challenging Examples

In [ ]:
# Get indices of bottom 3 examples
bottom_indices = np.argsort(individual_rouge_scores)[:3]

print("\n" + "="*100)
print("CHALLENGING EXAMPLES (Low ROUGE-L Scores)")
print("="*100)

for rank, idx in enumerate(bottom_indices, 1):
    test_example = dataset_dict['test'][idx]

    print(f"\n\n{'='*100}")
    print(f"EXAMPLE {rank} - ROUGE-L: {individual_rouge_scores[idx]:.4f}")
    print(f"{'='*100}")

    print(f"\nDIALOGUE (truncated):")
    dialogue = test_example['dialogue']
    print(dialogue[:400] + "..." if len(dialogue) > 400 else dialogue)

    print(f"\n{'-'*100}")
    print(f"REFERENCE SOAP:")
    print(references[idx])

    print(f"\n{'-'*100}")
    print(f"GENERATED SOAP:")
    print(predictions[idx])

### 7.8 Visualization

In [ ]:
# Create visualizations
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Distribution of ROUGE-L scores
axes[0, 0].hist(individual_rouge_scores, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].axvline(np.mean(individual_rouge_scores), color='red', linestyle='--',
                   label=f'Mean: {np.mean(individual_rouge_scores):.3f}')
axes[0, 0].set_xlabel('ROUGE-L Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of ROUGE-L Scores')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# 2. Summary length comparison
pred_lengths = [len(p.split()) for p in predictions]
ref_lengths = [len(r.split()) for r in references]

axes[0, 1].scatter(ref_lengths, pred_lengths, alpha=0.5, s=20)
axes[0, 1].plot([0, max(ref_lengths)], [0, max(ref_lengths)], 'r--', label='Perfect match')
axes[0, 1].set_xlabel('Reference Summary Length (words)')
axes[0, 1].set_ylabel('Generated Summary Length (words)')
axes[0, 1].set_title('Summary Length Comparison')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# 3. SOAP component presence
components = list(component_presence.keys())
counts = list(component_presence.values())
percentages = [(c/total_predictions)*100 for c in counts]

bars = axes[1, 0].bar(components, percentages, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[1, 0].set_ylabel('Percentage of Summaries (%)')
axes[1, 0].set_title('SOAP Component Presence in Generated Summaries')
axes[1, 0].set_ylim([0, 105])
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 2,
                    f'{pct:.1f}%', ha='center', va='bottom', fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# 4. Metrics comparison
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BERTScore F1']
scores = [
    rouge_results['rouge1'],
    rouge_results['rouge2'],
    rouge_results['rougeL'],
    F1.mean().item()
]

bars = axes[1, 1].barh(metrics, scores, color=['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4'])
axes[1, 1].set_xlabel('Score')
axes[1, 1].set_title('Model Performance Metrics')
axes[1, 1].set_xlim([0, 1])
for bar, score in zip(bars, scores):
    width = bar.get_width()
    axes[1, 1].text(width + 0.02, bar.get_y() + bar.get_height()/2.,
                    f'{score:.3f}', ha='left', va='center', fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Visualization saved as 'evaluation_results.png'")

### 7.9 Save Results

In [ ]:
# Save evaluation results
evaluation_results = {
    'model_name': MODEL_NAME,
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'rouge_scores': {
        'rouge1': float(rouge_results['rouge1']),
        'rouge2': float(rouge_results['rouge2']),
        'rougeL': float(rouge_results['rougeL']),
        'rougeLsum': float(rouge_results['rougeLsum'])
    },
    'bertscore': {
        'precision': float(P.mean()),
        'recall': float(R.mean()),
        'f1': float(F1.mean())
    },
    'soap_component_coverage': {
        k: f"{(v/total_predictions)*100:.1f}%"
        for k, v in component_presence.items()
    },
    'statistics': {
        'mean_rouge_l': float(np.mean(individual_rouge_scores)),
        'std_rouge_l': float(np.std(individual_rouge_scores)),
        'max_rouge_l': float(max(individual_rouge_scores)),
        'min_rouge_l': float(min(individual_rouge_scores))
    }
}

# Save to JSON
with open('evaluation_results.json', 'w') as f:
    json.dump(evaluation_results, f, indent=2)

print("✓ Evaluation results saved to 'evaluation_results.json'")

# Save predictions
results_df = pd.DataFrame({
    'dialogue': [dataset_dict['test'][i]['dialogue'] for i in range(len(predictions))],
    'reference_soap': references,
    'generated_soap': predictions,
    'rouge_l_score': individual_rouge_scores
})

results_df.to_csv('detailed_predictions.csv', index=False)
print("✓ Detailed predictions saved to 'detailed_predictions.csv'")

# Copy to Google Drive
!cp evaluation_results.json "/content/drive/MyDrive/model/"
!cp evaluation_results.png "/content/drive/MyDrive/model/"
!cp detailed_predictions.csv "/content/drive/MyDrive/model/"
print("✓ Results copied to Google Drive")

## 8. Final Summary

In [ ]:
print("\n" + "="*100)
print("FINAL EVALUATION SUMMARY")
print("="*100)

print("\n1. MODEL CONFIGURATION:")
print(f"   - Base Model: {MODEL_NAME}")
print(f"   - Fine-tuning Method: LoRA (Parameter-Efficient)")
print(f"   - Training Samples: {len(dataset_dict['train']):,}")
print(f"   - Validation Samples: {len(dataset_dict['validation']):,}")
print(f"   - Test Samples: {len(dataset_dict['test']):,}")

print("\n2. TRAINING RESULTS:")
print(f"   - Training Loss: {train_result.training_loss:.4f}")
print(f"   - Training Time: {train_result.metrics['train_runtime']/3600:.2f} hours")

print("\n3. QUANTITATIVE RESULTS:")
print(f"   - ROUGE-1: {rouge_results['rouge1']:.4f}")
print(f"   - ROUGE-2: {rouge_results['rouge2']:.4f}")
print(f"   - ROUGE-L: {rouge_results['rougeL']:.4f}")
print(f"   - BERTScore F1: {F1.mean():.4f}")

print("\n4. SOAP COMPONENT COVERAGE:")
for component, count in component_presence.items():
    percentage = (count / total_predictions) * 100
    print(f"   - {component.capitalize()}: {percentage:.1f}%")



